In [1]:
import kagglehub
import pandas as pd
import numpy as np
import json

/Users/shauryasingru/Desktop/ML Projects/movie-recommender-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download(
                            "tmdb/tmdb-movie-metadata",
                            output_dir = '../data')
movies_path =  path + "/tmdb_5000_movies.csv"
credits_path = path + "/tmdb_5000_credits.csv"

In [3]:
movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

In [4]:
movies_full = pd.merge(left = movies, right = credits, left_on = 'id', right_on = 'movie_id').reset_index(drop = True)

In [5]:
movies_df = movies_full[['title_x', 'overview', 'keywords', 'genres', 'cast', 'crew']]
movies_df = movies_df.rename({'title_x': 'title'}, axis = 1)

In [6]:
indices = (
            movies_df['title']
            .reset_index()
            .set_index('title')
           )
indices

,index
title,
Avatar,0
Pirates of the Caribbean: At World's End,1
Spectre,2
The Dark Knight Rises,3
John Carter,4
...,...
El Mariachi,4798
Newlyweds,4799
"Signed, Sealed, Delivered",4800


In [7]:
movies_df.head()

,title,overview,keywords,genres,cast,crew
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [8]:
from ast import literal_eval

features = ['keywords', 'genres', 'cast', 'crew']

for feature in features:
    movies_df[feature] = movies_df[feature].apply(literal_eval)

In [9]:
movies_df.keywords

0       [{'id': 1463, 'name': 'culture clash'}, {'id':...
1       [{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...
2       [{'id': 470, 'name': 'spy'}, {'id': 818, 'name...
3       [{'id': 849, 'name': 'dc comics'}, {'id': 853,...
4       [{'id': 818, 'name': 'based on novel'}, {'id':...
                              ...                        
4798    [{'id': 5616, 'name': 'united states–mexico ba...
4799                                                   []
4800    [{'id': 248, 'name': 'date'}, {'id': 699, 'nam...
4801                                                   []
4802    [{'id': 1523, 'name': 'obsession'}, {'id': 224...
Name: keywords, Length: 4803, dtype: object

In [10]:
keyword_counts = {}
genre_counts = {}

for movie in movies_df['keywords']:
    for kw_data in movie:
        keyword = kw_data['name']
        keyword_counts[keyword] = keyword_counts.get(keyword, 0) + 1

for movie in movies_df['genres']:
    for genre_data in movie:
        genre = genre_data['name']
        genre_counts[genre] = genre_counts.get(genre, 0) + 1

In [11]:
movies_df.loc[0, 'keywords']

[{'id': 1463, 'name': 'culture clash'},
 {'id': 2964, 'name': 'future'},
 {'id': 3386, 'name': 'space war'},
 {'id': 3388, 'name': 'space colony'},
 {'id': 3679, 'name': 'society'},
 {'id': 3801, 'name': 'space travel'},
 {'id': 9685, 'name': 'futuristic'},
 {'id': 9840, 'name': 'romance'},
 {'id': 9882, 'name': 'space'},
 {'id': 9951, 'name': 'alien'},
 {'id': 10148, 'name': 'tribe'},
 {'id': 10158, 'name': 'alien planet'},
 {'id': 10987, 'name': 'cgi'},
 {'id': 11399, 'name': 'marine'},
 {'id': 13065, 'name': 'soldier'},
 {'id': 14643, 'name': 'battle'},
 {'id': 14720, 'name': 'love affair'},
 {'id': 165431, 'name': 'anti war'},
 {'id': 193554, 'name': 'power relations'},
 {'id': 206690, 'name': 'mind and soul'},
 {'id': 209714, 'name': '3d'}]

In [12]:
keyword_counts['culture clash']

14

In [108]:
def top_keywords(row, length = 3):
    kw_counts = {kw['name'].replace(' ', ''): keyword_counts[kw['name']] for kw in row}
    kw_counts = sorted(kw_counts, key = lambda kw: kw_counts[kw], reverse = True)
    kw = [keyword for keyword in kw_counts]

    if len(kw) > length:
        return kw[:length]
    else:
        return kw
    
def top_genres(row, length, all = True):
    gen_counts = {genre['name'].replace(' ', ''): genre_counts[genre['name']] for genre in row}
    gen_counts = sorted(gen_counts, key = lambda gen: gen_counts[gen], reverse=True)
    genres = [genre for genre in gen_counts]

    if all:
        return genres
    
    if len(genres) > length:
        return genres[:length]
    else:
        return genres
    
    
def top_cast(row, length):
    cast = [actor['name'].lower().replace(' ', '') for actor in row]
    cast = cast[:length]
    return cast

def director(row):
    director = [crew['name'].lower().replace(' ', '') for crew in row if crew['job'] == 'Director']
    return director

In [109]:
movies_df['top_keywords'] = movies_df['keywords'].apply(top_keywords, length = 6)
movies_df['top_cast'] = movies_df['cast'].apply(top_cast, length = 5)
movies_df['director'] = movies_df['crew'].apply(director)
movies_df['top_genres'] = movies_df['genres'].apply(top_genres, length = 3, all = True)
movies_df.rename({'title_x': 'title'}, axis = 1, inplace=True)

In [110]:
top_keywords_all = set()
for movie in movies_df['top_keywords']:
    top_keywords_all.update(movie)

top_keywords_cleaned = set()
for term in top_keywords_all:
    if ' ' in term:
        words = term.split(' ')
        top_keywords_cleaned.update(words)
    else:
        top_keywords_cleaned.add(term)

print(len(top_keywords_cleaned))

4801


In [111]:
top_genres_all = set()
for movie in movies_df['top_genres']:
    top_genres_all.update(movie)

top_genres_cleaned = set()
for term in top_genres_all:
    if ' ' in term:
        words = term.split(' ')
        top_genres_cleaned.update(words)
    else:
        top_genres_cleaned.add(term)

print(len(top_genres_cleaned))

20


In [69]:
set(movies_df['top_keywords'][0])

{'3d', 'alien', 'battle', 'future', 'soldier', 'space'}

In [112]:
movies_df.loc[(movies_df.title == 'Iron Man'), 'top_keywords'].to_list()

[['aftercreditsstinger',
  'superhero',
  'basedoncomicbook',
  'marvelcomic',
  'marvelcinematicuniverse',
  'middleeast']]

In [113]:
def create_soup(x):
    soup = x['top_keywords'] + x['top_cast'] + x['director'] + x['top_genres']
    return ' '.join(soup)

soup_df = pd.DataFrame(movies_df['title'])
soup_df['soup'] = movies_df.apply(create_soup, axis=1)
soup_df = soup_df.set_index('title')

In [114]:
soup_df.loc['Iron Man','soup']

'aftercreditsstinger superhero basedoncomicbook marvelcomic marvelcinematicuniverse middleeast robertdowneyjr. terrencehoward jeffbridges shauntoub gwynethpaltrow jonfavreau Action Adventure ScienceFiction'

In [115]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

count = CountVectorizer(stop_words='english')
count_matrix = normalize(count.fit_transform(soup_df['soup']), norm = 'l2')
cosine_sim_count = cosine_similarity(count_matrix, count_matrix)

In [118]:
count_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 63331 stored elements and shape (4803, 16846)>

In [117]:
soup_words_all = set()
for movie in soup_df['soup']:
    movie_kws = movie.split()
    soup_words_all.update(movie_kws)
print(len(soup_words_all))

16509


In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(soup_df['soup'])
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [87]:
dir(tfidf_matrix)
tfidf_matrix.indices

array([  14,  460, 5538, ..., 8101, 1978, 1951],
      shape=(63335,), dtype=int32)

In [119]:
def get_recommendations(title, cosine_sim):
    idx = indices.loc[title].values[0]
    sim_scores = enumerate(cosine_sim[idx])

    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return movies_df['title'].iloc[movie_indices]

def get_recommendations_multi_input(movies, cosine_sim):
    sim_scores = []
    input_movie_indices = []
    for title in movies:
        idx = indices.loc[title].values[0]
        input_movie_indices.append(idx)
        sim_scores.append(cosine_sim[idx])

    weights = [3, 2, 1]

    sim_scores = np.average(np.array(sim_scores), axis = 0, weights = weights)
    
    sim_scores = enumerate(normalize(sim_scores.reshape(1,-1), norm='l2').flatten())

    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

    movie_indices = [i[0] for i in sim_scores if i[0] not in input_movie_indices][:10]

    # return movie_indices

    return indices.iloc[movie_indices].reset_index()['title']

In [131]:
movie = "12 Years a Slave"

display(
    # get_recommendations(movie, cosine_sim_count),
    get_recommendations(movie, cosine_sim_tfidf)
    )


3430                         Shame
188                           Salt
3750              The Great Escape
1051                     Prisoners
2522            The Imitation Game
979            Free State of Jones
912     Interview with the Vampire
877                     Black Mass
2507                     Slow Burn
1211                       Amistad
Name: title, dtype: object

In [ ]:
movies = [
            'Housefull',
            'Babel',
            'Iron Man',
]

display(
# get_recommendations_multi_input(movies, cosine_sim_count),
get_recommendations_multi_input(movies, cosine_sim_tfidf)
)

In [ ]:
indices = movies_df[['title', 'overview']].reset_index().set_index('title')

In [ ]:
import joblib
import os

os.makedirs('../artifacts', exist_ok=True)

cosine_sim_path = '../artifacts/cosine_sim.pkl'
indices_path = '../artifacts/indices.pkl'

joblib.dump(cosine_sim_tfidf.astype('float32'), cosine_sim_path)
joblib.dump(indices, indices_path)